# AgriRakshak cloud training

Trains a MobileNetV3-Small classifier for bell pepper, potato, tomato, and an out-of-scope plant class. Images come from the PlantVillage color dataset. The workflow preserves its leaf-grouped test split, creates a leaf-grouped validation split, evaluates confidence, and exports the exact ONNX bundle required by the API.

**Important:** PlantVillage uses controlled backgrounds. Final exhibition claims require separate testing on real phone/field photographs.

**Runtime note:** GPU acceleration is optional. Run the one-epoch smoke test on CPU if no GPU is available. A free Colab GPU may be used for the longer baseline run, but availability is variable and no paid runtime is required.

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('NVIDIA GPU detected:')
    print(result.stdout)
else:
    print('No NVIDIA GPU detected. Continuing in CPU mode for the smoke test.')
    print('For the longer baseline run, restart with a free Colab GPU if one is available.')

In [ ]:
# Pin the training environment to a reviewed repository commit.
!git clone --depth 1 --branch main https://github.com/kaalakhatta/AGRIRAKSHAK.git
%cd AGRIRAKSHAK
!git checkout fcb4913
%cd ml
!python -m pip install -q -e '.[cloud]'

In [ ]:
from pathlib import Path
WORK = Path('/content/agrirakshak-training')
DATASET = WORK / 'dataset'
RUN = WORK / 'baseline-v1'
BUNDLE = WORK / 'bundle'
WORK.mkdir(parents=True, exist_ok=True)
print(WORK)

In [ ]:
!agrirakshak-prepare-plantvillage --output {DATASET} --max-per-class 1600 --unsupported-per-source-class 120
!cat {DATASET / 'split-manifest.summary.json'}

In [ ]:
# One-epoch smoke test proves the complete data and training path first.
# This is intentionally small and can run on CPU if no free GPU is available.
!agrirakshak-train --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --output {WORK / 'smoke'} --epochs 1 --batch-size 32 --workers 2 --freeze-features --class-balanced

In [ ]:
# Final baseline. A free Colab GPU is recommended but not required.
# If the session expires or compute is unavailable, preserve the smoke-test results
# and resume later rather than treating an unexecuted final run as completed.
!agrirakshak-train --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --output {RUN} --epochs 15 --batch-size 64 --workers 2 --class-balanced

In [ ]:
!agrirakshak-evaluate --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --checkpoint {RUN / 'best.pt'} --output {RUN / 'evaluation'} --batch-size 64 --workers 2
!agrirakshak-export --checkpoint {RUN / 'best.pt'} --evaluation {RUN / 'evaluation' / 'metrics.json'} --output {BUNDLE} --version baseline-v1
!ls -lh {BUNDLE}

In [ ]:
import json
metrics = json.loads((RUN / 'evaluation' / 'metrics.json').read_text())
print('Test macro F1:', metrics['test']['macro_f1'])
print('Confidence threshold:', metrics['confidence_policy']['threshold'])
print('Coverage:', metrics['confidence_policy']['coverage'])
print('Review per-class metrics and the confusion matrix before deploying.')

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/agrirakshak-baseline-v1', 'zip', BUNDLE)
files.download(archive)
# Also download evidence needed for review.
evidence = shutil.make_archive('/content/agrirakshak-evaluation', 'zip', RUN / 'evaluation')
files.download(evidence)